# SwinFusion (Swin-transformer infrared + visible fusion)

Fusion method for the `fusion/` pipeline. Save this notebook as `fusion/methods/transformers/swinfusion.ipynb`.

- **Input:** `fusion/data/<SOURCE>/` with `rgb/` + `ir/` image sequences (same filenames, frame-aligned), or `visible.mp4` + `thermal.mp4`.
- **Output:** `fusion/out/swinfusion_<SOURCE>.mp4` (grayscale fused video, same resolution and frame count as the input).
- **Clips and speed:** `SOURCES` lists the clips (default: the three Blender forest test clips). `FRAME_STEP = 1`
  (default) fuses every frame, as rfn_nest does; use a GPU. On CPU (about 48 s per 1080p frame), `FRAME_STEP = 10`
  fuses only the frames `FUSION_EVALUATION.ipynb` scores (every 10th) and repeats each one: same scores, but the
  video updates only 3 times per second.
- **Method:** SwinFusion from [Linfeng-Tang/SwinFusion](https://github.com/Linfeng-Tang/SwinFusion). Code and the released infrared-visible weights are cloned on first run into `fusion/methods/_vendor/SwinFusion/`. Needs the `timm` package (installed automatically if missing).

Locally: set `SOURCES`, select the `.venv` kernel, run all cells. On **Colab** (T4 GPU): `forest_batch_1.zip` must be on Drive at `DRIVE_DATA`, as for `rfn_nest_blender`; the clips are unzipped into `/content` and the videos are written to `DRIVE_OUT/out`. Then score the result with `fusion/FUSION_EVALUATION.ipynb`.

In [10]:
METHOD = 'swinfusion'        # output file: fusion/out/<METHOD>_<SOURCE>.mp4

SOURCES = ['forest_01_night-moon_nadir-hover',   # folder names inside fusion/data/ (default: the three Blender
           'forest_07_afternoon_steep-push',     # forest test clips, the same ones the rfn_nest_blender notebooks
           'forest_10_dusk_high-orbit']          # hold out)

# Colab only (same Drive layout as rfn_nest_blender_v*.ipynb); locally fusion/data and fusion/out are used
DATASET = 'forest_batch_1'
DRIVE_DATA = '/content/drive/MyDrive/Vidhance Extended View/Blender renders/batch_1'   # .zip, or the forest_batch_1 folder
DRIVE_OUT = '/content/drive/MyDrive/Vidhance Extended View/models_output/transformers'           # videos go to DRIVE_OUT/out

SCALE = 1                     # fuse at this fraction of the input size (output is resized back to full size)
MAX_FRAMES = None             # None = all frames, or an int to fuse only the first N
FRAME_STEP = 1                # fuse every n-th frame and repeat it until the next one (keeps the frame count).
                              # 10 = exactly the frames FUSION_EVALUATION scores (EVAL_STRIDE = 10), so the scores equal
                              # a full run at a tenth of the cost; 1 = fuse every frame (a smooth video to watch)
FPS = 30.0                    # frame rate for image-sequence inputs (videos keep their own)
OVERWRITE = True              # False = skip clips whose output already exists
PREVIEW_FRAME = 100           # frame index shown in the preview at the end (a multiple of FRAME_STEP)


SWIN_CKPT = None              # None -> the repo's IR-visible weights (10000_E.pth, the EMA weights its test script loads)
NORMALIZE_PER_FRAME = False   # SwinFusion's own test script does NOT min-max normalize; True can flicker

In [ ]:
import os, sys, shutil, subprocess, time, zipfile, importlib
from itertools import islice
from pathlib import Path

def _need(mod, pip_name=None):                       # fallback if requirements.txt was not installed
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or mod], check=True)

for _m, _p in [('numpy', None), ('cv2', 'opencv-python'), ('torch', None), ('timm', None), ('tqdm', None), ('matplotlib', None)]:
    _need(_m, _p)

import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:                                         # same layout as rfn_nest_blender_v*.ipynb
    drive.mount('/content/drive')
    if not torch.cuda.is_available():
        raise RuntimeError('No GPU: choose Runtime > Change runtime type > T4 GPU, then run all cells again.')
    VENDOR = Path('/content/_vendor')
    BATCH_ROOT = Path('/content/data') / DATASET     # forest_batch_1 unzipped from Drive
    DATA_DIR = Path('/content/fusion_data')          # forest_<clip>/{rgb,ir,clear} built from it below
    OUT_DIR = Path(DRIVE_OUT) / 'out'
else:
    here = Path.cwd().resolve()
    fusion = next((d / 'fusion' for d in [here, *here.parents] if (d / 'fusion' / 'data').is_dir()),
                  next((d for d in [here, *here.parents] if d.name == 'fusion' and (d / 'data').is_dir()), None))
    if fusion is None:
        raise FileNotFoundError('Could not find fusion/data from %s. Start Jupyter inside the repo.' % here)
    VENDOR = fusion / 'methods' / '_vendor'          # third-party code is cloned here (add to .gitignore)
    DATA_DIR, OUT_DIR = fusion / 'data', fusion / 'out'

STREAMS = ('visible_haze', 'visible_clear', 'thermal_haze')
VIS_W, VIS_H, TH_Y0, TH_Y1 = 1280, 720, 76, 436      # thermal -> visible registration, as in rfn_nest_blender

def prepare_colab_sources():
    """Unzips forest_batch_1 from Drive (first run only) and writes forest_<clip>/{rgb,ir,clear} for SOURCES,
    the same files rfn_nest_blender exports: rgb = visible_haze, clear = visible_clear, ir = registered thermal_haze."""
    clips = [s[len('forest_'):] for s in SOURCES if s.startswith('forest_')]
    complete = lambda: all((BATCH_ROOT / c / 'frames' / s).is_dir() for c in clips for s in STREAMS)
    if BATCH_ROOT.is_dir() and not complete():
        print('The copy in %s is incomplete (interrupted unzip?), replacing it.' % BATCH_ROOT)
        shutil.rmtree(BATCH_ROOT)
    if not BATCH_ROOT.is_dir():
        src, t0 = Path(DRIVE_DATA), time.time()
        if not src.exists():
            raise FileNotFoundError('%s not found. Upload %s to Drive or change DRIVE_DATA.' % (src, DATASET))
        BATCH_ROOT.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            print('copying the folder from Drive, this can take a long time for this many files...')
            shutil.copytree(src, BATCH_ROOT)
        else:
            with zipfile.ZipFile(src) as zf:
                zf.extractall(BATCH_ROOT.parent)
        print('data ready in %.0f s' % (time.time() - t0))
    for clip in clips:
        frames, dst = BATCH_ROOT / clip / 'frames', DATA_DIR / ('forest_' + clip)
        names = sorted(p.name for p in (frames / 'visible_haze').iterdir() if p.suffix == '.png')
        if (dst / 'ir').is_dir() and len(list((dst / 'ir').iterdir())) == len(names):
            continue
        dst.mkdir(parents=True, exist_ok=True)
        for sub, stream in (('rgb', 'visible_haze'), ('clear', 'visible_clear')):
            if not (dst / sub).exists():
                (dst / sub).symlink_to(frames / stream)
        (dst / 'ir').mkdir(exist_ok=True)
        for name in tqdm(names, desc=clip, leave=False):
            th = cv2.imread(str(frames / 'thermal_haze' / name), cv2.IMREAD_GRAYSCALE)
            cv2.imwrite(str(dst / 'ir' / name), cv2.resize(th[TH_Y0:TH_Y1], (VIS_W, VIS_H), interpolation=cv2.INTER_LINEAR))
        print('prepared', dst)

if IN_COLAB:
    prepare_colab_sources()

def fetch_repo(name, url):
    d = VENDOR / name
    if not d.is_dir():
        VENDOR.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', url, str(d)], check=True)
    return d

def pick_ckpt(user_value, default):
    p = Path(user_value) if user_value else Path(default)
    if not p.exists():
        raise FileNotFoundError('Checkpoint not found: %s (set the *_CKPT option in the config cell)' % p)
    return p

print('device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('outputs:', OUT_DIR)
print('sources:', sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
copying the folder from Drive, this can take a long time for this many files...


In [ ]:
SWIN_REPO = fetch_repo('SwinFusion', 'https://github.com/Linfeng-Tang/SwinFusion.git')

# models/network_swinfusion1.py imports from timm.models.layers, which newer timm releases deprecate
# in favour of timm.layers (same helpers). Point the import at the new location when it exists.
net_py = SWIN_REPO / 'models' / 'network_swinfusion1.py'
src = net_py.read_text()
try:
    import timm.layers
    if 'from timm.models.layers import' in src:
        net_py.write_text(src.replace('from timm.models.layers import', 'from timm.layers import'))
except ImportError:
    pass

def use_repo(d):
    """Puts repo d first on the import path and forgets modules imported from another vendored repo, so that
    same-named packages (DATFuse and ITFuse both have `Networks`, others `models`, `utils`) cannot be mixed up
    when several of these notebooks run in one kernel."""
    d = str(d)
    while d in sys.path:
        sys.path.remove(d)
    sys.path.insert(0, d)
    for name, mod in list(sys.modules.items()):
        locs = [getattr(mod, '__file__', None) or ''] + [str(p) for p in (getattr(mod, '__path__', None) or [])]
        if any(str(VENDOR) in l for l in locs) and not any(l.startswith(d) for l in locs if l):
            del sys.modules[name]

use_repo(SWIN_REPO)
print(sorted(p.name for p in SWIN_REPO.iterdir()))

In [ ]:
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

use_repo(SWIN_REPO)                                   # in case another notebook ran in this kernel
from models.network_swinfusion1 import SwinFusion

DEFAULT_CKPT = SWIN_REPO / 'Model' / 'Infrared_Visible_Fusion' / 'Infrared_Visible_Fusion' / 'models' / '10000_E.pth'
swin_ckpt = pick_ckpt(SWIN_CKPT, DEFAULT_CKPT)
print(swin_ckpt)

# same hyper-parameters as define_model() in the repo's test_swinfusion.py
swin_model = SwinFusion(upscale=1, in_chans=1, img_size=128, window_size=8, img_range=1.,
                        depths=[6, 6, 6, 6], embed_dim=60, num_heads=[6, 6, 6, 6],
                        mlp_ratio=2, upsampler=None, resi_connection='1conv')
state = torch.load(swin_ckpt, map_location='cpu')
swin_model.load_state_dict(state['params'] if 'params' in state else state, strict=True)
swin_model.eval().to(DEVICE)

In [ ]:
@torch.no_grad()
def fuse_pair(vis_gray, ir_gray):
    # SwinFusion expects float images in [0, 1]; input order is (IR, VIS-Y) as in the repo's test script.
    # The model reflect-pads to a multiple of the window size (8) internally and crops back.
    vi = torch.from_numpy(vis_gray.astype(np.float32) / 255.0)[None, None].to(DEVICE)
    ir = torch.from_numpy(ir_gray.astype(np.float32) / 255.0)[None, None].to(DEVICE)
    out = swin_model(ir, vi)[0, 0].float().cpu().numpy() * 255.0
    if NORMALIZE_PER_FRAME:
        out = (out - out.min()) / (out.max() - out.min() + 1e-5) * 255
    return np.clip(np.round(out), 0, 255).astype(np.uint8)

def reset_state():                                     # this method keeps no state between frames
    pass

In [ ]:
# Reading a scene from fusion/data/<SOURCE>/
import re
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
RGB_DIRS = ('rgb', 'visible', 'vis', 'vi', 'color')
IR_DIRS = ('ir', 'thermal', 'infrared', 't')
RGB_VIDEOS = ('visible.mp4', 'rgb.mp4')
IR_VIDEOS = ('thermal.mp4', 'ir.mp4')

def rescale(img):
    return img if SCALE == 1 else cv2.resize(img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)

def _natural(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', p.name)]

def _find(d, names, want_dir):
    for p in sorted(d.iterdir()):
        if p.name.lower() in names and (p.is_dir() if want_dir else p.is_file()):
            return p

def image_pairs(rgb_dir, ir_dir):
    rgb = {p.name: p for p in sorted(rgb_dir.iterdir(), key=_natural) if p.suffix.lower() in IMG_EXT}
    ir = {p.name: p for p in sorted(ir_dir.iterdir(), key=_natural) if p.suffix.lower() in IMG_EXT}
    names = sorted(set(rgb) & set(ir), key=lambda n: _natural(Path(n)))
    if names:
        pairs = [(rgb[n], ir[n]) for n in names]
    else:
        print('WARNING: rgb/ and ir/ share no filenames, pairing by sorted position')
        a, b = sorted(rgb.values(), key=_natural), sorted(ir.values(), key=_natural)
        pairs = list(zip(a, b))
    def gen():
        for a, b in pairs:
            v, t = cv2.imread(str(a), cv2.IMREAD_COLOR), cv2.imread(str(b), cv2.IMREAD_GRAYSCALE)
            if v is None or t is None:
                raise IOError('Could not read %s / %s' % (a, b))
            yield v, t
    return len(pairs), FPS, gen()

def video_pairs(rgb_path, ir_path):
    a, b = cv2.VideoCapture(str(rgb_path)), cv2.VideoCapture(str(ir_path))
    if not (a.isOpened() and b.isOpened()):
        raise IOError('Could not open %s / %s' % (rgb_path, ir_path))
    n = min(int(a.get(cv2.CAP_PROP_FRAME_COUNT)), int(b.get(cv2.CAP_PROP_FRAME_COUNT)))
    fps = a.get(cv2.CAP_PROP_FPS) or FPS
    def gen():
        try:
            while True:
                ok1, v = a.read(); ok2, t = b.read()
                if not (ok1 and ok2):
                    break
                yield v, cv2.cvtColor(t, cv2.COLOR_BGR2GRAY)
        finally:
            a.release(); b.release()
    return n, fps, gen()

def open_pairs(src_dir):
    rgb_dir, ir_dir = _find(src_dir, RGB_DIRS, True), _find(src_dir, IR_DIRS, True)
    if rgb_dir and ir_dir:
        return image_pairs(rgb_dir, ir_dir)
    rgb_v, ir_v = _find(src_dir, RGB_VIDEOS, False), _find(src_dir, IR_VIDEOS, False)
    if rgb_v and ir_v:
        return video_pairs(rgb_v, ir_v)
    raise FileNotFoundError('%s needs rgb/ + ir/ folders or visible.mp4 + thermal.mp4, found: %s'
                            % (src_dir, sorted(p.name for p in src_dir.iterdir())))

def make_writer(path, fps, size):
    for code in ('avc1', 'mp4v'):                   # H.264 if this OpenCV build has it, else MPEG-4 (an 'avc1 / Failed to initialize VideoWriter' log line is harmless)
        w = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*code), fps, size)
        if w.isOpened():
            return w
    raise RuntimeError('OpenCV could not open a video writer for %s' % path)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
gray = lambda bgr: cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
previews = {}
for SOURCE in SOURCES:
    src_dir = DATA_DIR / SOURCE
    if not src_dir.is_dir():
        raise FileNotFoundError('fusion/data/%s not found. Available: %s'
                                % (SOURCE, sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())))
    out_path = OUT_DIR / ('%s_%s.mp4' % (METHOD, SOURCE))
    if out_path.exists() and not OVERWRITE:
        print('exists, skipping:', out_path)
        continue
    n_total, fps, frames = open_pairs(src_dir)
    if MAX_FRAMES:
        n_total = min(n_total, MAX_FRAMES) if n_total else MAX_FRAMES
    reset_state()                                      # per-clip state of the output normalization, if any
    tmp_path = OUT_DIR / ('.%s_%s.partial.mp4' % (METHOD, SOURCE))   # renamed only when finished
    writer, done, fused, g, t0 = None, 0, 0, None, time.time()
    try:
        for i, (rgb, ir) in enumerate(tqdm(islice(frames, MAX_FRAMES), total=n_total or None, desc=SOURCE)):
            H, W = rgb.shape[:2]
            if i % FRAME_STEP == 0:
                if ir.shape[:2] != (H, W):
                    ir = cv2.resize(ir, (W, H))
                g = fuse_pair(rescale(gray(rgb)), rescale(ir))
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
                if g.shape[:2] != (H, W):
                    g = cv2.resize(g, (W, H), interpolation=cv2.INTER_LINEAR)
                fused += 1
            if writer is None:
                writer = make_writer(tmp_path, fps, (W, H))
            writer.write(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR))
            if i == PREVIEW_FRAME or (SOURCE not in previews and i == (n_total or 1) - 1):
                previews[SOURCE] = (rgb, ir, g, i)
            done += 1
    finally:
        if writer is not None:
            writer.release()
        frames.close()
    if done == 0:
        raise RuntimeError('No frames were read from %s' % src_dir)
    os.replace(tmp_path, out_path)
    sec = time.time() - t0
    print('%s: %d frames written, %d fused in %.1fs (%.2f s per fused frame on %s) -> %s'
          % (SOURCE, done, fused, sec, sec / fused, DEVICE, out_path.name))

In [ ]:
for SOURCE, (rgb, ir, g, i) in previews.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, img, title in zip(axes, [cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB), ir, g], ['visible', 'thermal', 'fused: ' + METHOD]):
        ax.imshow(img, cmap=None if img.ndim == 3 else 'gray', vmin=0, vmax=255); ax.set_title('%s (%s, frame %d)' % (title, SOURCE, i)); ax.axis('off')
    plt.show()
print('Next: run fusion/FUSION_EVALUATION.ipynb (once per SOURCE) to score', METHOD)